In [3]:
# Importing rewuired libraries.
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# Loading the feature-engineered dataset.
df = pd.read_csv("../data/processed/energydata_features.csv", parse_dates=["date"], index_col="date")

# Defining target and features.
target = "Appliances"
features = df.drop(columns=[target]).columns.tolist()

X = df[features]
y = df[target]


# Train-test split (time-based)

split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Training models.

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, objective="reg:squarederror")
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}

    # Saving model using pickle.
    with open(f"../models/{name}_model.pkl", "wb") as f:
        pickle.dump(model, f)

    print(f"{name} - MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.3f}")


# Saving results for evaluation notebook.

results_df = pd.DataFrame(results).T
results_df.to_csv("../outputs/reports/model_results.csv")
print("\nModel evaluation results saved to ../outputs/reports/model_results.csv")

# Saving the feature importance for RandomForest and XGBoost models.
print("\nAll models trained, evaluated, and saved using pickle.")



Training LinearRegression...
LinearRegression - MAE: 0.00, RMSE: 0.00, R2: 1.000

Training RandomForest...
RandomForest - MAE: 3.90, RMSE: 14.36, R2: 0.975

Training XGBoost...
XGBoost - MAE: 4.07, RMSE: 12.52, R2: 0.981

Model evaluation results saved to ../outputs/reports/model_results.csv

All models trained, evaluated, and saved using pickle.
